In [1]:
library(tidyverse)
library(limma)
library(grid)
library(gridExtra)

source("../../evaluation_utils/plots_eda.R")
source("../../evaluation_utils/filtering.R")

Warning message:
“package ‘dplyr’ was built under R version 4.5.3”
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘gridExtra’


The following object is masked from ‘package:dplyr’:

    combine



Attaching package: ‘data.table’


The following objects are masked from ‘package:lubridate’:

    hour, isoweek, isoyear, mday, minute, month, quarter, second, wday,
    week, yday, year


The following objects are masked from ‘package:dplyr’:

    between, first, last


The following object is masked fro

## Load data

**Prerequisite:** Run `prepare_ccRCC_data.py` first to create:
- `before/central_intensities_log_UNION.tsv` (outer-joined union matrix)
- Updated `design.tsv` files in each site folder (with `Condition` column)

In [2]:
# Paths (relative to evaluation_data/ccRCC_studies/)
path_before <- "before/"
path_after  <- "after/"

# Load union matrix produced by prepare_ccRCC_data.py
intensities <- read.csv(
    paste0(path_before, "central_intensities_log_UNION.tsv"),
    sep = "\t", header = TRUE, row.names = 1, check.names = FALSE
)
cat("Union matrix loaded:", nrow(intensities), "features x", ncol(intensities), "samples\n")

# Build metadata from tracked site design files.
sites <- c("PDC000127", "PXD030344", "PXD042844")
metadata <- map_dfr(sites, function(site) {
    design <- read.csv(
        paste0(path_before, site, "/design.tsv"),
        sep = "\t", header = TRUE, check.names = FALSE
    )
    sample_col <- if ("Sample" %in% colnames(design)) "Sample" else colnames(design)[1]
    condition <- if ("Tumor" %in% colnames(design)) {
        ifelse(as.integer(design$Tumor) == 1L, "Tumor", "Normal")
    } else if ("Condition" %in% colnames(design)) {
        ifelse(design$Condition %in% c(1, "1", "Tumor"), "Tumor", "Normal")
    } else {
        stop(paste("No condition columns found in", site, "design.tsv"))
    }
    tibble(file = design[[sample_col]], condition = condition, lab = site)
}) %>% column_to_rownames("file")
metadata$file <- rownames(metadata)
cat("Metadata:", nrow(metadata), "samples\n")
print(table(metadata$lab, metadata$condition))

Union matrix loaded: 13826 features x 887 samples
Metadata: 887 samples
           
            Normal Tumor
  PDC000127     84   110
  PXD030344    232   232
  PXD042844    114   115


In [3]:
# Align matrix columns to metadata row order
shared_samples <- intersect(colnames(intensities), metadata$file)
cat("Shared samples between matrix and metadata:", length(shared_samples), "\n")

intensities <- intensities[, shared_samples]
metadata    <- metadata[shared_samples, ]
cat("Matrix after alignment:", nrow(intensities), "features x", ncol(intensities), "samples\n")

Shared samples between matrix and metadata: 887 
Matrix after alignment: 13826 features x 887 samples


## Filter features

Uses `filter_per_center` from `filtering.R` with `min_samples = 5` (matching batches + covariates logic).

`drop_row = TRUE` means a feature is removed if it has fewer than `min_samples` non-NA values in **any** site — the same hard-filter logic used in other datasets.


In [4]:
MIN_SAMPLES <- 5L   # integer literal — matches min_samples: 2 in config.yml

cat("Features before filtering:", nrow(intensities), "\n")

intensities_filtered <- filter_per_center(
    intensities,
    metadata,
    quantitative_column_name = "file",
    centers                  = unique(metadata$lab),
    center_column_name       = "lab",
    min_samples              = MIN_SAMPLES,
    drop_row                 = TRUE
)

cat("Features retained:", nrow(intensities_filtered), "\n")
cat("Features removed: ", nrow(intensities) - nrow(intensities_filtered), "\n")


Features before filtering: 13826 


Filtering by lab - min 5 not-NA per lab

	Before filtering: 13826 887

	After filtering: 6282 887



Features retained: 6282 
Features removed:  7544 


Also filter by condition group: drop features with fewer than `MIN_SAMPLES` non-NA values in either Tumor or Normal samples (across all sites combined).


In [5]:
intensities_filtered <- filter_per_center(
    intensities_filtered,
    metadata,
    quantitative_column_name = "file",
    centers                  = unique(metadata$condition),
    center_column_name       = "condition",
    min_samples              = MIN_SAMPLES,
    drop_row                 = TRUE
)

cat("Features retained after condition filter:", nrow(intensities_filtered), "\n")


Filtering by condition - min 5 not-NA per condition

	Before filtering: 6282 887

	After filtering: 6282 887



Features retained after condition filter: 6282 


## Central batch effect correction (limma removeBatchEffect)

In [6]:
# Build design matrix: Condition as covariate (Tumor vs Normal)
metadata <- metadata %>%
    mutate(condition = factor(condition, levels = c("Normal", "Tumor")))

design <- model.matrix(~ condition, data = metadata)
colnames(design) <- c("Intercept", "Tumor")
print(head(design, 5))

              Intercept Tumor
CPT0002370001         1     0
CPT0000790001         1     0
CPT0088570001         1     0
CPT0065820001         1     0
CPT0012920003         1     0


In [7]:
# Run central limma removeBatchEffect
# Batch = Dataset (PDC000127, PXD030344, PXD042844)
intensities_corrected <- limma::removeBatchEffect(
    as.matrix(intensities_filtered),
    batch   = metadata$lab,
    design  = design
) %>% as.data.frame()

cat("Corrected matrix:", nrow(intensities_corrected), "features x",
    ncol(intensities_corrected), "samples\n")

Corrected matrix: 6282 features x 887 samples


## Save results

In [8]:
if (!dir.exists(path_after)) dir.create(path_after, recursive = TRUE)

write.table(
    intensities_corrected %>% rownames_to_column("Gene"),
    file = paste0(path_after, "intensities_log_Rcorrected_UNION.tsv"),
    sep = "\t", quote = FALSE, row.names = FALSE, col.names = TRUE
)
cat("Saved:", paste0(path_after, "intensities_log_Rcorrected_UNION.tsv"), "\n")

Saved: after/intensities_log_Rcorrected_UNION.tsv 


## Session info

In [9]:
sessionInfo()

R version 4.5.2 (2025-10-31)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 24.04.4 LTS

Matrix products: default
BLAS/LAPACK: /home/yuliya-cosybio/miniforge3/envs/fedRBE/lib/libopenblasp-r0.3.30.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=en_US.UTF-8       LC_NUMERIC=C              
 [3] LC_TIME=en_US.UTF-8        LC_COLLATE=en_US.UTF-8    
 [5] LC_MONETARY=en_US.UTF-8    LC_MESSAGES=en_US.UTF-8   
 [7] LC_PAPER=en_US.UTF-8       LC_NAME=C                 
 [9] LC_ADDRESS=C               LC_TELEPHONE=C            
[11] LC_MEASUREMENT=en_US.UTF-8 LC_IDENTIFICATION=C       

time zone: Europe/Berlin
tzcode source: system (glibc)

attached base packages:
[1] grid      stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] data.table_1.18.2.1 viridis_0.6.5       viridisLite_0.4.3  
 [4] ggsci_4.2.0         umap_0.2.10.0       patchwork_1.3.2    
 [7] gridExtra_2.3       limma_3.66.0        lubridate_1.9.5    
[10] forcats_1.0